In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from shutil import rmtree
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
from openpyxl.styles import Font,PatternFill,Border,Side,Alignment
OutputFolder=Path('/mnt/data/Final') if Path('/mnt/data').exists() else Path('Final')
TempFolder=Path('/mnt/data/TempPictures') if Path('/mnt/data').exists() else Path('TempPictures')
if OutputFolder.exists():
    rmtree(OutputFolder)
if TempFolder.exists():
    rmtree(TempFolder)
OutputFolder.mkdir()
TempFolder.mkdir()
Hours=8000
YearHours=8760
GasEF=0.181
GridEF=0.153183
WindMW=1.5
WindCF=0.291
BoilerEff=0.98
CHPFuel=1573.43
WHRUSaving=157.34
WindMWh=WindMW*YearHours*WindCF
DirectWind=2294.24
BoilerInput=WindMWh-DirectWind
RenewableSteam=BoilerInput*BoilerEff
SteamSupport=RenewableSteam*3600/Hours
WHRUCO2=WHRUSaving*GasEF
DirectWindCO2=DirectWind*GridEF
SteamCO2=RenewableSteam*GasEF
TotalCO2=WHRUCO2+DirectWindCO2+SteamCO2
CAPEX=2530000
Rate=0.08
Life=25
CRF=Rate*(1+Rate)**Life/(((1+Rate)**Life)-1)
AnnualCAPEX=CAPEX*CRF
FixedOPEX=129460
AnnualCost=AnnualCAPEX+FixedOPEX
ElecPrice=120
GasPrice=45
CarbonPrice=75
BenefitNoCO2=DirectWind*ElecPrice+WHRUSaving*GasPrice+RenewableSteam*GasPrice
BenefitWithCO2=BenefitNoCO2+TotalCO2*CarbonPrice
NetNoCO2=BenefitNoCO2-AnnualCost
NetWithCO2=BenefitWithCO2-AnnualCost
AvoidNoCO2=(AnnualCost-BenefitNoCO2)/TotalCO2
AvoidWithCO2=(AnnualCost-BenefitWithCO2)/TotalCO2
CashNoCO2=BenefitNoCO2-FixedOPEX
CashWithCO2=BenefitWithCO2-FixedOPEX
PaybackNoCO2=CAPEX/CashNoCO2
PaybackWithCO2=CAPEX/CashWithCO2
PVFactor=(1-(1+Rate)**(-Life))/Rate
NPVNoCO2=CashNoCO2*PVFactor-CAPEX
NPVWithCO2=CashWithCO2*PVFactor-CAPEX
BCRNoCO2=(CashNoCO2*PVFactor)/CAPEX
BCRWithCO2=(CashWithCO2*PVFactor)/CAPEX
Utility=pd.DataFrame([
['Electricity',1449.92,1228.34,-221.58,'Surplus'],
['HP steam',6497.05,6309.55,-187.49,'Almost balanced'],
['MP steam',4111.36,18698.62,14587.26,'Main deficit'],
['LP steam',722.24,10842.95,10120.71,'Large deficit'],
['Cooling water',0,52987.02,52987.02,'High cooling burden']],columns=['Utility','Supply MJ/h','Demand MJ/h','Net deficit MJ/h','Meaning'])
Chapter5=pd.DataFrame([
['CHP baseline fuel',CHPFuel,'MWh/y'],
['WHRU/HRSG fuel saving',WHRUSaving,'MWh/y'],
['WHRU/HRSG CO2 saving',WHRUCO2,'tCO2/y'],
['Wind electricity',WindMWh,'MWh/y'],
['Direct wind use',DirectWind,'MWh/y'],
['Electricity to electrode boiler',BoilerInput,'MWh/y'],
['Renewable steam',RenewableSteam,'MWh/y'],
['Steam support',SteamSupport,'MJ/h'],
['Total CO2 saving',TotalCO2,'tCO2/y'],
['Old screening case',79.74,'tCO2/y']],columns=['Indicator','Value','Unit'])
CHP=pd.DataFrame([
['WHRU/HRSG heat recovery',WHRUSaving,'MWh/y',WHRUCO2,'Direct fuel saving'],
['Direct wind electricity',DirectWind,'MWh/y',DirectWindCO2,'Avoided grid electricity'],
['Electricity to electrode boiler',BoilerInput,'MWh/y',0,'Power-to-steam input'],
['Renewable steam',RenewableSteam,'MWh/y',SteamCO2,'Avoided fossil steam'],
['Total CHP Utility-Hub Retrofit',WindMWh+WHRUSaving,'MWh/y',TotalCO2,'Final result']],columns=['Component','Energy','Unit','CO2 saving t/y','Note'])
Tradeoff=pd.DataFrame([
['CO2 reduction','tCO2/y',0,TotalCO2,80,20,TotalCO2+100,300,1000],
['Renewable electricity share','%',0,140,0,0,140,30,150],
['Steam flexibility','MJ/h',0,SteamSupport,0,0,SteamSupport,300,1500],
['Material circularity','Score',0.25,0.25,0.80,0.30,0.85,0.60,1.00],
['E-factor improvement','Score',0.10,0.10,0.70,0.20,0.75,0.50,1.00],
['Water circularity','Score',0.20,0.20,0.25,0.80,0.85,0.60,1.00],
['Network integration','Score',0.35,0.70,0.55,0.50,0.90,0.60,1.00],
['Financial feasibility','Score',0.80,0.65,0.55,0.50,0.55,0.50,1.00],
['Implementation simplicity','Score',0.90,0.60,0.55,0.55,0.40,0.40,1.00],
['Policy alignment','Score',0.40,0.90,0.85,0.85,0.95,0.70,1.00]],columns=['Criteria','Unit','Baseline','CHP Retrofit','Material','Water','Full System','Min target','Max target'])
EnergyCO2=pd.DataFrame([
['Wind turbine capacity',WindMW,'MW'],
['Wind electricity',WindMWh,'MWh/y'],
['Direct wind use',DirectWind,'MWh/y'],
['Electricity to electrode boiler',BoilerInput,'MWh/y'],
['Electrode boiler efficiency',BoilerEff,'-'],
['Renewable steam',RenewableSteam,'MWh/y'],
['Steam support',SteamSupport,'MJ/h'],
['WHRU/HRSG CO2 saving',WHRUCO2,'tCO2/y'],
['Direct wind CO2 saving',DirectWindCO2,'tCO2/y'],
['Renewable steam CO2 saving',SteamCO2,'tCO2/y'],
['Total CO2 saving',TotalCO2,'tCO2/y']],columns=['Indicator','Value','Unit'])
Economics=pd.DataFrame([
['CAPEX',CAPEX,'EUR'],
['Annualized CAPEX',AnnualCAPEX,'EUR/y'],
['Fixed OPEX',FixedOPEX,'EUR/y'],
['Annual cost',AnnualCost,'EUR/y'],
['Benefit without CO2 price',BenefitNoCO2,'EUR/y'],
['Benefit with CO2 price',BenefitWithCO2,'EUR/y'],
['Net without CO2 price',NetNoCO2,'EUR/y'],
['Net with CO2 price',NetWithCO2,'EUR/y'],
['Avoidance cost without CO2 price',AvoidNoCO2,'EUR/tCO2'],
['Avoidance cost with CO2 price',AvoidWithCO2,'EUR/tCO2'],
['Payback without CO2 price',PaybackNoCO2,'years'],
['Payback with CO2 price',PaybackWithCO2,'years'],
['NPV without CO2 price',NPVNoCO2,'EUR'],
['NPV with CO2 price',NPVWithCO2,'EUR'],
['BCR without CO2 price',BCRNoCO2,'-'],
['BCR with CO2 price',BCRWithCO2,'-']],columns=['Indicator','Value','Unit'])
TONF=pd.DataFrame([
['Technical','Check steam pressure, boiler size and exhaust heat','Steam pressure or temperature mismatch','Engineering check'],
['Infrastructural','Add WHRU/HRSG tie-in, grid connection and steam line','Limited connection capacity','Tie-in and grid study'],
['Organisational','Share data, costs and operation rules','Unclear ownership','Shared agreement'],
['Network','Connect energy, material and water layers','High CHP dependency','Backup and maintenance'],
['Financial','Control CAPEX and use carbon benefits','Price and CAPEX sensitivity','Supplier quote and sensitivity']],columns=['TONF','Need','Risk','Action'])
def save_table_excel(path,sheets):
    with pd.ExcelWriter(path,engine='openpyxl') as writer:
        for name,table in sheets.items():
            table.to_excel(writer,sheet_name=name,index=False)
    wb=load_workbook(path)
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.font=Font(bold=True,color='FFFFFF')
            cell.fill=PatternFill('solid',fgColor='1F4E78')
            cell.alignment=Alignment(horizontal='center')
        for row in ws.iter_rows():
            for cell in row:
                cell.border=Border(left=Side(style='thin',color='D9D9D9'),right=Side(style='thin',color='D9D9D9'),top=Side(style='thin',color='D9D9D9'),bottom=Side(style='thin',color='D9D9D9'))
                cell.alignment=Alignment(wrap_text=True,vertical='center')
                if isinstance(cell.value,float):
                    cell.number_format='0.00'
        for col in ws.columns:
            width=max(len(str(c.value)) if c.value is not None else 0 for c in col)+2
            ws.column_dimensions[col[0].column_letter].width=min(width,32)
        ws.freeze_panes='A2'
    wb.save(path)
def add_picture(path,sheet_name,pictures):
    wb=load_workbook(path)
    ws=wb.create_sheet(sheet_name)
    ws['A1']='Pictures'
    ws['A1'].font=Font(bold=True,size=14)
    row=3
    for title,pic in pictures:
        ws[f'A{row}']=title
        ws[f'A{row}'].font=Font(bold=True)
        img=Image(pic)
        img.width=720
        img.height=420
        ws.add_image(img,f'A{row+1}')
        row+=24
    wb.save(path)
def make_chapter5_pictures():
    p1=TempFolder/'ch5_utility.png'
    p2=TempFolder/'ch5_co2.png'
    plt.figure(figsize=(8,4.5))
    plt.bar(Utility['Utility'],Utility['Net deficit MJ/h'])
    plt.axhline(0,color='black',linewidth=1)
    plt.title('Chapter 5 utility balance')
    plt.ylabel('MJ/h')
    plt.xticks(rotation=25,ha='right')
    plt.tight_layout()
    plt.savefig(p1,dpi=200)
    plt.close()
    plt.figure(figsize=(8,4.5))
    names=['WHRU','Direct wind','Renewable steam']
    vals=[WHRUCO2,DirectWindCO2,SteamCO2]
    plt.bar(names,vals)
    plt.title('Chapter 5 CO2 saving components')
    plt.ylabel('tCO2/y')
    plt.tight_layout()
    plt.savefig(p2,dpi=200)
    plt.close()
    return [('Utility balance',p1),('CO2 saving components',p2)]
def make_chapter6_pictures():
    p1=TempFolder/'ch6_spider.png'
    p2=TempFolder/'ch6_tradeoff.png'
    labels=['CO2','Renewable','Steam','Material','E-factor','Water','Network','Finance','Simplicity','Policy']
    baseline=[0.05,0.05,0.10,0.25,0.10,0.20,0.35,0.80,0.90,0.40]
    retrofit=[0.85,1.00,0.75,0.25,0.10,0.20,0.70,0.65,0.60,0.90]
    full=[0.95,1.00,0.80,0.85,0.75,0.85,0.90,0.55,0.40,0.95]
    angles=np.linspace(0,2*np.pi,len(labels),endpoint=False).tolist()
    angles+=angles[:1]
    plt.figure(figsize=(8,8))
    ax=plt.subplot(111,polar=True)
    for values,name in [(baseline,'Baseline'),(retrofit,'CHP Retrofit'),(full,'Full System')]:
        values=values+values[:1]
        ax.plot(angles,values,label=name)
        ax.fill(angles,values,alpha=0.10)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels,fontsize=8)
    ax.set_ylim(0,1)
    ax.legend(loc='upper right',bbox_to_anchor=(1.25,1.10))
    plt.title('Chapter 6 spider diagram')
    plt.tight_layout()
    plt.savefig(p1,dpi=200)
    plt.close()
    plt.figure(figsize=(9,5))
    plt.bar(['CHP Retrofit','Material','Water','Full System'],[TotalCO2,80,20,TotalCO2+100])
    plt.title('Chapter 6 CO2 trade-off')
    plt.ylabel('tCO2/y')
    plt.tight_layout()
    plt.savefig(p2,dpi=200)
    plt.close()
    return [('Spider diagram',p1),('CO2 trade-off',p2)]
Chapter5Path=OutputFolder/'Chapter5.xlsx'
Chapter6Path=OutputFolder/'Chapter6.xlsx'
save_table_excel(Chapter5Path,{'Summary':Chapter5,'Utility':Utility,'CHP':CHP})
add_picture(Chapter5Path,'Pictures',make_chapter5_pictures())
save_table_excel(Chapter6Path,{'Summary':EnergyCO2,'Tradeoff':Tradeoff,'Economics':Economics,'TONF':TONF})
add_picture(Chapter6Path,'Pictures',make_chapter6_pictures())
rmtree(TempFolder)
print('Done')
print(Chapter5Path)
print(Chapter6Path)

Done
Final\Chapter5.xlsx
Final\Chapter6.xlsx
